# SAC Irrigation Training — v2.8.0 (Colab Pro)

**Architecture:** CTDE + VDN Factorized Critic  
**Hyperparameters:** `ent_coef=0.05` (fixed), `max_grad_norm=1.0`, LR decay

## Key changes from v2.7
| # | Change | Effect |
|---|--------|--------|
| 1 | `x1_overshoot_norm` added as 9th per-agent feature | Direct gradient signal from r6 reward; addresses wet-year x1-conditioning weakness |
| 2 | Episode-length curriculum (60d → 93d at step 50k) | Reduces critic-target variance during initial value-function learning |
| 3 | OBS_DIM 1097 → 1227 | per-agent block 8 → 9 features |
| 4 | TOTAL_TIMESTEPS 500k → 250k (default) | v2.7 peaked at step 200k on both seeds; 250k captures peak with half the compute |

## Before running
1. `Runtime → Change runtime type → A100 GPU` (Colab Pro). L4 is the next best.
2. Add WandB API key as a Colab Secret: left sidebar → key icon → `+ Add new secret` → Name: `WANDB_API_KEY` → toggle **Notebook access ON**.
3. Mount Google Drive in Cell 1.

## What to watch on WandB
| Metric | Expected v2.8 behaviour | Abort if |
|--------|------------------------|----------|
| `train/critic_loss` | Stays bounded (< 100) throughout training — curriculum should prevent the v2.7 explosion | Exceeds 500 after step 100k |
| `train/ent_coef` | Constant at 0.05 | Deviates |
| `rollout/ep_len_mean` | 60 for first ~50k steps, then jumps to 93 | Never reaches 93 (curriculum stuck) |
| `rollout/ep_rew_mean` | Improves over time, peak likely near step 200k | Monotonically decreasing after step 100k |

## Seed plan (v2.8 paired-samples design)
**Use the SAME seeds as the v2.7 baseline** (0 and 1 first, then 2, 3, 4 to extend the sample).
This is a *paired-samples* design: comparing v2.8_seed0 vs v2.7_seed0 directly answers "did the protocol change help on this initialization?", and the same comparison across seeds 0 and 1 answers "does it help across initializations?".
The v2.7 baseline results are saved in `results/legacy/runs_legacy_sac_vdn_2.6/` and `results/runs/sac_perfect_det_*_seed{0,1}.json` — they are NOT overwritten by v2.8 training (v2.8 writes to `results/rl/sac_v28_seed{N}/`).

- Session 1 → `SEED = 0`  (paired with v2.7 seed 0)
- Session 2 → `SEED = 1`  (paired with v2.7 seed 1)
- Session 3 → `SEED = 2`  (new seed, extends sample to N=3)
- Session 4 → `SEED = 3`  (new seed, extends sample to N=4)
- Session 5 → `SEED = 4`  (new seed, extends sample to N=5)

Results saved to `MyDrive/thesis_results/sac_v28_seed{N}_{timestamp}/` (Colab) or `/kaggle/working/results_v28_seed{N}/` (Kaggle).

## Estimated runtimes (v2.8 obs is ~12% larger than v2.7; offset by 250k cap → similar total time)
| Hardware | Steps/sec | 250k steps |
|----------|-----------|------------|
| A100 | ~110–150 | ~28–38 min |
| L4 | ~70–90 | ~46–60 min |
| T4 | ~50–70 | ~60–85 min |

In [ ]:
# ── CELL 1: Mount Drive, clone repo, install deps ───────────────────────────
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'✓  Drive mounted. Results → {DRIVE_ROOT}')

if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True
)
os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')
print('✓  Repo cloned')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'wandb>=0.16',
    'pytest',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3, torch
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'torch:             {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')
torch.set_num_threads(4)

In [ ]:
# ── CELL 2: WandB secret + GPU check ────────────────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('✓  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'⚠  Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('   Training continues without WandB — add key to Colab Secrets to enable.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed — no GPU allocated')

In [ ]:
# ── CELL 3: Pre-training validation (MUST PASS before Cell 4) ───────────────
#
# Runs smoke tests + VDN unit tests + curriculum tests + bench.
# DO NOT proceed to Cell 4 if any test fails.

import subprocess, sys, time

print('Running smoke tests (includes new x1_overshoot + curriculum tests)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'
print()

print('Running VDN unit tests (includes v2.7 and v2.6 legacy load guards)...')
r2 = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r2.returncode == 0, 'VDN UNIT TESTS FAILED'
print()

print('Benchmarking step rate...')
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import CTDESACPolicy, make_sac_policy_kwargs

bench_env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])
bench_model = SAC(
    policy=CTDESACPolicy, env=bench_env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=5_000, batch_size=256, learning_starts=500,
    ent_coef=0.05, verbose=0, seed=0,
)
bench_model.learn(total_timesteps=300)
t0 = time.time()
bench_model.learn(total_timesteps=500, reset_num_timesteps=False)
rate = 500 / (time.time() - t0)
del bench_model, bench_env

print(f'Step rate:    {rate:.1f} steps/sec')
print(f'Est. 250k:    {250_000 / rate / 60:.0f} min  ({250_000 / rate / 3600:.1f} h)')
print()

from src.rl.gym_env import IrrigationEnv, OBS_DIM
env_check = IrrigationEnv(randomize=False)
obs_check, _ = env_check.reset()
assert obs_check.shape[0] == 1227, f'Expected obs_dim=1227, got {obs_check.shape[0]}'
print(f'obs_dim:      {obs_check.shape[0]} ✓  (9 features × 130 agents + 9 scalars + 48 forecast)')
# Verify x1_overshoot feature at slot 8
overshoot_at_reset = obs_check[:1170].reshape(130, 9)[:, 8]
print(f'x1_overshoot at reset:  max={overshoot_at_reset.max():.4f} (should be ~0)')
print()
print('✓  ALL VALIDATION PASSED — safe to proceed to Cell 4')

In [ ]:
# ── CELL 4: Training (250k steps, ~30-60 min on A100) ────────────────────────
#
# v2.8 training uses 250k steps by default (down from v2.7's 500k) because
# both v2.7 seeds peaked at step 200k.  The 250k cap captures the peak with
# half the compute.
#
# Change SEED per session.  Use SAME seeds as v2.7 baseline (0, 1, 2, 3, 4)
# for paired-samples comparison.  v2.7 results in results/runs/sac_perfect_*
# are NOT overwritten — v2.8 writes to results/rl/sac_v28_seed{N}/.

SEED = 0   # ← CHANGE THIS per session: 0, 1, 2, 3, 4 (paired with v2.7 baseline)

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,             # v2.8 default
    curriculum_warmup_steps=50_000,      # v2.8 curriculum default
    curriculum_short_len=60,             # short-episode length
)
print(f'\n✓  Training complete — seed {SEED}, v2.8')

In [ ]:
# ── CELL 5: Copy results to Google Drive ────────────────────────────────────
import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_v28_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_v28_seed{SEED}_{timestamp}'

def ignore_replay_buffer(directory, files):
    return [f for f in files if 'replay_buffer' in f]

shutil.copytree(src, dst, ignore=ignore_replay_buffer)

total_mb, file_count = 0, 0
for root, dirs, files in os.walk(dst):
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        total_mb += size / 1e6
        file_count += 1
        rel = os.path.relpath(os.path.join(root, f), dst)
        print(f'  {rel}  ({size/1e6:.2f} MB)')

print(f'\n✓  {file_count} files ({total_mb:.1f} MB) saved to:')
print(f'   {dst}')

In [ ]:
# ── CELL 6: Quick post-training diagnostics ──────────────────────────────────
#
# v2.8 targets (vs v2.7 baselines):
#   spatial_std    > 0.01   (v2.7 ~0.02 — should be similar/better)
#   corr(u, rain)  < 0      (v2.7 -0.49 wet/100% — should be similar)
#   corr(u, x1)    < 0      (v2.7 +0.05 wet/100% — v2.8 should now SHOW response!)
#   waterlog days < 60      (v2.7 was ~76; targeting < 60 with new feature)
#   ep_len         == 93    (curriculum off at eval time)

import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv

best_model_path = f'/content/thesis/results/rl/sac_v28_seed{SEED}/best_model/best_model'

eval_env = DummyVecEnv([lambda: IrrigationEnv(randomize=False, curriculum_warmup_steps=0)])
loaded = SAC.load(best_model_path, env=eval_env)

obs = eval_env.reset()
all_actions, all_rain, all_x1 = [], [], []
ep_len = 0
done = False
while not done:
    action, _ = loaded.predict(obs, deterministic=True)
    all_actions.append(action[0].copy())
    raw_obs = obs[0]
    # x1 mean across agents (feature 0 of 9)
    x1_vals = raw_obs[:1170].reshape(130, 9)[:, 0]
    all_x1.append(x1_vals.mean())
    # rain is scalar block position 1174 (1170 + scalar[4])
    all_rain.append(float(raw_obs[1174]))
    obs, reward, done, info = eval_env.step(action)
    ep_len += 1
    done = done[0]

all_actions = np.array(all_actions)
u_arr      = all_actions.mean(axis=1) * 12
spatial_std = all_actions.mean(axis=0).std()
corr_rain   = float(np.corrcoef(u_arr, np.array(all_rain))[0, 1])
corr_x1     = float(np.corrcoef(u_arr, np.array(all_x1))[0, 1])

print('=== v2.8 Post-training Diagnostics (dry/100% scenario) ===')
print(f'Episode length:         {ep_len}  (expected 93)')
print(f'Mean irrigation mm/day: {u_arr.mean():.2f}')
print(f'Spatial std:            {spatial_std:.4f}  (target > 0.01)')
print(f'corr(u, rain_today):    {corr_rain:+.3f}  (target < 0; v2.7 was -0.41 in dry/100%)')
print(f'corr(u, x1_mean):       {corr_x1:+.3f}  (target < 0; v2.7 was +0.18 in dry/100%)')
print()
checks = [
    (ep_len == 93,       f'ep_len {ep_len} == 93'),
    (spatial_std > 0.01, f'spatial_std {spatial_std:.4f} > 0.01'),
    (corr_rain < 0,      f'corr(u,rain) {corr_rain:+.3f} < 0'),
    (corr_x1   < 0,      f'corr(u,x1)   {corr_x1:+.3f} < 0  (v2.8 PRIMARY TARGET)'),
]
for ok, label in checks:
    print(f'  {"✓" if ok else "⚠"} {label}')

In [ ]:
# ── CELL 7: Resume from Drive checkpoint (if session was interrupted) ────────
# Fill in CHECKPOINT_STEP and CHECKPOINT_DRIVE_PATH, then uncomment and run.

# SEED = 2
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_v28_seed{SEED}_YYYYMMDD_HHMMSS'
#
# import shutil, os
# local_dir = f'/content/thesis/results/rl/sac_v28_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from stable_baselines3 import SAC
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
# from src.rl.train import _make_lr_schedule, LR_START, LR_END, TOTAL_TIMESTEPS
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])  # curriculum already done
# ckpt_zip = f'{local_dir}/checkpoints/sac_v28_seed{SEED}_{CHECKPOINT_STEP}_steps'
# model = SAC.load(ckpt_zip, env=env)
# model.lr_schedule = _make_lr_schedule(LR_START, LR_END)
#
# remaining = TOTAL_TIMESTEPS - CHECKPOINT_STEP
# print(f'Resuming from step {CHECKPOINT_STEP}, {remaining} steps remaining...')
# model.learn(total_timesteps=remaining, reset_num_timesteps=False, progress_bar=True)
# model.save(f'{local_dir}/sac_v28_seed{SEED}_final')
# print('Resume complete.')